# 03 - HAAQI Score
Compute HAAQI score for the baseline output.

**Before this, run `00_shared_drive_data.ipynb`:** the clone-Clarity cell and **Part A** (builds the valid-set dataset from the shared Drive zips). It puts Clarity in `/content/clarity` and the dataset in `/content/cadenza_data/cad2/task2`. (`01_setup.ipynb` can replace the Clarity part, but not the dataset.)

Note: this notebook uses paths relative to `/content`, so a Colab session restart means running 00 again.


1. Check prerequisites (nothing is downloaded here)
Stops with a clear message if notebook 00 has not been run in this session.

In [31]:
import os
from pathlib import Path
os.chdir('/content')

root = Path('/content/cadenza_data/cad2/task2')
clarity_dir = Path('/content/clarity')
meta = root / 'metadata'

need = {
    'Clarity baseline (run 00 clone cell or 01_setup)': clarity_dir / 'recipes/cad2/task2/baseline/evaluate.py',
    'valid metadata (run 00 Part A)': meta / 'scenes.valid.json',
    'valid audio (run 00 Part A)': root / 'audio/valid',
}
missing = [k for k, p in need.items() if not p.exists()]
assert not missing, 'Missing: ' + '; '.join(missing)
print('prerequisites ok')

prerequisites ok


In [32]:
meta = root / 'metadata'
for name in ['scenes.valid.json', 'scene_listeners.valid.json', 'music.valid.json', 'listeners.valid.json']:
    assert (meta / name).exists(), f'missing {name}'
print('valid audio folders:', sorted(p.name for p in (root / 'audio' / 'valid').iterdir())[:5], '...')

valid audio folders: ['Calcante_ed_Achille_001', 'Marche_001', 'Marche_002', 'Precipitoso_nel_mar_001', 'Precipitoso_nel_mar_002'] ...


2. Pick a first scene in the valid set with its first listener

In [33]:
import json

scenes = json.load(open(meta / 'scenes.valid.json'))
scene_listeners = json.load(open(meta / 'scene_listeners.valid.json'))

scene_id = next(iter(scene_listeners))
listener_id = scene_listeners[scene_id][0]
print('scene   :', scene_id, scenes[scene_id])
print('listener:', listener_id)

one = root / 'metadata_one_scene'
one.mkdir(exist_ok=True)
json.dump({scene_id: scenes[scene_id]}, open(one / 'scenes.json', 'w'))
json.dump({scene_id: [listener_id]}, open(one / 'scene_listeners.json', 'w'))

scene   : S50001 {'music': 'anitrasdance_001', 'gain': 'gain_0236'}
listener: L5051


3. Run the baseline - Same as notebook 02 but on the valid set.

In [34]:
exp = Path('exp_valid_one_scene').resolve()
overrides = f'path.root={root} path.scenes_file={one}/scenes.json path.scene_listeners_file={one}/scene_listeners.json path.exp_folder={exp}'

!PYTHONPATH={clarity_dir} python clarity/recipes/cad2/task2/baseline/enhance.py {overrides}

[2026-11-12 18:19:24,481][__main__][INFO] - Loading model cadenzachallenge/ConvTasNet_Bassoon_NonCausal
[2026-11-12 18:19:24,661][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/cadenzachallenge/ConvTasNet_Bassoon_NonCausal/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-11-12 18:19:24,662][huggingface_hub.utils._http][WARNING] - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[2026-11-12 18:19:24,678][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cadenzachallenge/ConvTasNet_Bassoon_NonCausal/5828031665ea5ce9d8b05ec71e53c7eb68fe0761/config.json?%2Fcadenzachallenge%2FConvTasNet_Bassoon_NonCausal%2Fresolve%2Fmain%2Fconfig.json=&etag=%22a049a1a444f751a3571e04e1d01697c68e99788e%22 "HTTP/1.1 200 OK"
[2026-11-12 18:19:24,697][httpx][INFO] - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/cadenzachallenge/ConvTasNet_Bassoon_N

4. Compute HAAQI - Rebuilds the reference from the stems and then runs it through the listener's hearing-aid model, then compares it to our enhanced signal for each ear.

In [35]:
!PYTHONPATH={clarity_dir} python clarity/recipes/cad2/task2/baseline/evaluate.py {overrides}

[2026-11-12 18:23:54,795][__main__][INFO] - Evaluating from enhanced_signals directory
[2026-11-12 18:23:54,839][__main__][INFO] - [001/001] Evaluating S50001 for listener L5051
[2026-11-12 18:25:04,982][__main__][INFO] - Done!


5. Read the score - HAAQI  between 0 and 1, higher the better.

In [36]:
import csv

with open(exp / 'scores.csv') as f:
    rows = list(csv.DictReader(f))
for r in rows:
    print(f"scene {r['scene']} | listener {r['listener']} | song {r['song']}")
    print(f"left: {float(r['left_haaqi']):.4f}")
    print(f"right: {float(r['right_haaqi']):.4f}")
    print(f"avg: {float(r['avg_haaqi']):.4f}")

scene S50001 | listener L5051 | song anitrasdance_001
left: 0.4827
right: 0.4809
avg: 0.4818


## DONE
If a score printed, HAAQI is computed for the baseline output.